# BenePick RAG API Server (Colab)
Qwen3-4B + BGE-M3 + bge-reranker-v2-m3

**실행 전 확인사항:**
- 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- Google Drive에 프로젝트 폴더 업로드 완료 여부

## 1. Google Drive 마운트 및 프로젝트 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# 드라이브 내 프로젝트 경로 (필요시 수정)
PROJECT_PATH = '/content/drive/MyDrive/benepick'
os.chdir(PROJECT_PATH)
print(f'작업 경로: {os.getcwd()}')
print(os.listdir('.'))

## 2. 의존성 설치

In [ ]:
%%capture
!pip install fastapi uvicorn pydantic-settings python-dotenv
!pip install sentence-transformers FlagEmbedding faiss-cpu rank-bm25 kiwipiepy
!pip install langchain langchain-core langchain-community langchain-huggingface
!pip install transformers accelerate bitsandbytes
!pip install pyngrok

## 3. 환경변수 설정

In [ ]:
import os

# LLM: HuggingFace Qwen3-4B 사용
os.environ['LLM_PROVIDER'] = 'huggingface'
os.environ['HF_MODEL_ID'] = 'Qwen/Qwen3-4B'

# HuggingFace 캐시 경로 (Drive에 캐싱 → 재시작 시 재다운로드 방지)
os.environ['HF_HOME'] = '/content/drive/MyDrive/benepick/.cache/huggingface'

print('환경변수 설정 완료')

## 4. ngrok 설정 (외부 URL 생성)

In [ ]:
# ngrok 토큰 설정 (https://dashboard.ngrok.com 에서 발급)
NGROK_TOKEN = ''  # ← 여기에 토큰 입력

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN

# 포트 8001 터널 생성
tunnel = ngrok.connect(8001)
RAG_API_URL = tunnel.public_url
print(f'RAG API 외부 URL: {RAG_API_URL}')
print(f'백엔드 .env에 아래 값을 추가하세요:')
print(f'RAG_API_URL={RAG_API_URL}')

## 5. RAG API 서버 실행

In [ ]:
import subprocess, sys

# rag/ 디렉토리에서 uvicorn 실행
os.chdir(PROJECT_PATH)
!uvicorn rag.api:app --host 0.0.0.0 --port 8001